In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Ensure working directory is project root
current_path = Path.cwd()
if current_path.name == 'notebook':
    os.chdir(current_path.parent)

data_dir = Path('data/raw')
if not data_dir.exists():
    data_dir = Path('dataset/raw')

print(f"Active working directory: {Path.cwd()}")
print(f"Data directory: {data_dir.resolve()} (Exists: {data_dir.exists()})")

# Load column definitions from features CSV
features_file = data_dir / 'NUSW-NB15_features.csv'
features_df = pd.read_csv(features_file, encoding='latin1')

# Strip whitespace from column names to avoid formatting issues (e.g., 'ct_src_ ltm')
features = [str(name).strip() for name in features_df['Name']]

Active working directory: d:\school\Perkuliahan\Semester_5\MLOps
Data directory: D:\school\Perkuliahan\Semester_5\MLOps\data\raw (Exists: True)
Total features defined in metadata: 49
First 10 features: ['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl']
Last 5 features:  ['ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label']


In [ ]:
# Define explicit dtypes for columns known to contain mixed/string values
# to avoid pandas DtypeWarning and optimize memory usage
dtype_spec = {
    'sport': str,
    'dsport': str,
    'ct_ftp_cmd': str,
    'attack_cat': str,
    'Label': np.int64
}

print("Loading UNSW-NB15 dataset splits (df1 - df4)...")
df1 = pd.read_csv(data_dir / 'UNSW-NB15_1.csv', names=features, dtype=dtype_spec, low_memory=False)
df2 = pd.read_csv(data_dir / 'UNSW-NB15_2.csv', names=features, dtype=dtype_spec, low_memory=False)
df3 = pd.read_csv(data_dir / 'UNSW-NB15_3.csv', names=features, dtype=dtype_spec, low_memory=False)
df4 = pd.read_csv(data_dir / 'UNSW-NB15_4.csv', names=features, dtype=dtype_spec, low_memory=False)

dfs = {'df1': df1, 'df2': df2, 'df3': df3, 'df4': df4}
print("All 4 dataframes loaded successfully.")

Loading UNSW-NB15 dataset splits (df1 - df4)...
All 4 dataframes loaded successfully.


In [ ]:
df4.info()

<class 'pandas.DataFrame'>
RangeIndex: 440044 entries, 0 to 440043
Data columns (total 49 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   srcip             440044 non-null  str    
 1   sport             440044 non-null  str    
 2   dstip             440044 non-null  str    
 3   dsport            440044 non-null  str    
 4   proto             440044 non-null  str    
 5   state             440044 non-null  str    
 6   dur               440044 non-null  float64
 7   sbytes            440044 non-null  int64  
 8   dbytes            440044 non-null  int64  
 9   sttl              440044 non-null  int64  
 10  dttl              440044 non-null  int64  
 11  sloss             440044 non-null  int64  
 12  dloss             440044 non-null  int64  
 13  service           440044 non-null  str    
 14  Sload             440044 non-null  float64
 15  Dload             440044 non-null  float64
 16  Spkts             440044 non-nu

In [ ]:
# hexadecimal port number
df1[df1['dsport']=='0x20205321']

,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
82531,175.45.176.1,-,149.171.126.12,0x20205321,esp,INT,0.000000,200,0,254,...,0,12,12,4,4,2,1,4,NaN,0
82532,175.45.176.1,0,149.171.126.12,0x20205321,esp,INT,0.000004,200,0,254,...,0,12,12,4,4,2,3,4,NaN,0


In [ ]:
df_filt = df1.dropna()
df_filt

,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
20,175.45.176.3,21223,149.171.126.18,32780,udp,INT,0.000021,728,0,254,...,0,1,1,1,1,1,1,1,Exploits,1
21,175.45.176.2,23357,149.171.126.16,80,tcp,FIN,0.240139,918,25552,62,...,0,3,2,2,1,1,1,1,Exploits,1
22,175.45.176.0,13284,149.171.126.16,80,tcp,FIN,2.390390,1362,268,254,...,0,5,2,2,1,1,1,1,Reconnaissance,1
39,175.45.176.2,13792,149.171.126.16,5555,tcp,FIN,0.175190,8168,268,254,...,0,1,1,1,1,1,1,1,Exploits,1
40,175.45.176.2,26939,149.171.126.10,80,tcp,FIN,0.190600,844,268,254,...,0,3,1,1,1,1,1,1,Exploits,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186498,175.45.176.1,58463,149.171.126.15,179,tcp,FIN,31.461020,924,708,254,...,0,6,6,2,2,2,1,2,Fuzzers,1
186499,175.45.176.1,58460,149.171.126.15,179,tcp,FIN,57.995934,1362,536,254,...,0,6,6,2,2,2,1,2,Fuzzers,1
186562,175.45.176.1,58967,149.171.126.15,179,tcp,FIN,32.361782,1128,622,254,...,0,6,6,2,2,2,1,2,Fuzzers,1
186658,175.45.176.1,58485,149.171.126.15,179,tcp,FIN,0.476639,470,354,254,...,0,8,8,2,2,2,1,2,Fuzzers,1
